# AG News: attention-head faults in the output projection W_o

The crash fault of the original study zeroes a head's output before the heads
are concatenated and projected by `W_o`. Because

`W_o(concat(head_0..head_H)) = sum_h W_o[:, h*d_k:(h+1)*d_k] @ head_h + b_o`,

zeroing head *h*'s `d_k` columns of `W_o` removes exactly the same term. A crash
is therefore one particular **weight fault in W_o**. This notebook corrupts
those columns in other ways and compares the outcome with the crash.

It loads the checkpoint trained by `notebooks/base_transformer/ag_news.ipynb`
(no training here) and uses the same fixed evaluation subset. Results go to
`results/base_transformer/ag_news/wo_faults/`; the original results are not
touched. Method and outputs: `docs/wo_faults.md`.

| Step | Question |
|---|---|
| E0 | Does zeroing W_o columns reproduce the crash exactly? (sanity) |
| E1 | Single head, every head: how harmful are bit flips and stuck-at values compared with the crash, and do they hit the same heads? |
| E2 | Random multi-head faults on the same head sets as the crash study |
| D1 | Most critical heads: damage vs the **value** a weight is changed to |
| D2 | Most critical heads: damage vs the **original magnitude** of the changed weight |
| D3 | Most critical heads: damage vs the **float32 bit** that is flipped |
| D4 | Pooled: is damage explained by the size of the change alone? |


## Setup (imports, globals, data split and model copied from the original notebook)

In [1]:
import json, pathlib, sys, time

import numpy as np
import pandas as pd
import torch

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from faulttol import config as C
from faulttol import multihead, plots, saferegion, single_head
from faulttol.injection import collect_attention_stats_per_head, get_all_heads
from faulttol.runtime import configure_threads, device_report, select_device
from faulttol.training import make_loader

print("torch", torch.__version__, "| threads", configure_threads(),
      "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("root:", ROOT)

torch 2.14.0+cu126 | threads 8 | cuda build 12.6 | cuda available True
root: D:\Laboratory\fault-tolerance-models-and-task-comparison


In [2]:
RESULTS_DIR = ROOT / "results" / "base_transformer" / "ag_news"
MODEL_NAME = "base_transformer_agnews"
DATA_DIR = ROOT / "data" / "ag_news"
# NVIDIA GPU when one is visible; FT_DEVICE=cpu reproduces the CPU-only run
DEVICE = select_device()

# the sequence length must match the trained checkpoint
MAX_LEN = json.loads((RESULTS_DIR / "summary.json").read_text(encoding="utf-8"))["max_len"]
VOCAB_SIZE = 30_000
NUM_CLASSES = 4
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

MODEL_CONFIG = dict(C.BASE_PARAM, activation="relu")

TRAIN_CONFIG = {
    "loss": "CrossEntropyLoss",
    "optimizer": "AdamW",
    # EXP_A trains its synthetic task at 1e-4/1e-2; a from-scratch text encoder
    # needs a slightly larger step to converge inside a CPU epoch budget.
    "learning_rate": 3e-4,
    "weight_decay": 1e-2,
    "batch_size": 128,
    "epochs": C._env_int("FT_EPOCHS", 12),
    "patience": C._env_int("FT_PATIENCE", 3),
    "grad_clip": 1.0,
    "selection_metric": "roc_auc",
    "cosine_schedule": True,
}

# degradation variables (as in EXP_A)
CRIT_COL = "D_pos_mean"       # criticality signal
DEG_COL = "D_mean"            # degradation variable for the fault-severity analysis
SAFE_COL = "D_pos_mean"       # safe-region degradation variable
DEG_LABEL = "mean probability damage"
SAFE_LABEL = "mean positive damage"
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

print("device:", device_report(DEVICE))
print("MAX_LEN", MAX_LEN, "| epochs", TRAIN_CONFIG["epochs"])

device: cuda:0 NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB | sm_89 | torch cuda 12.6 | tf32 off
MAX_LEN 64 | epochs 12


In [3]:
import collections, re

TOKEN_RE = re.compile(r"[a-z0-9']+")

def read_split(name):
    df = pd.read_csv(DATA_DIR / ("%s.csv" % name), header=None,
                     names=["label", "title", "description"])
    text = (df["title"].fillna("") + " " + df["description"].fillna("")).str.lower()
    return text.tolist(), (df["label"].to_numpy() - 1).astype(np.int64)

train_text, train_y = read_split("train")
test_text, test_y = read_split("test")
print("raw: train=%d test=%d classes=%d" % (len(train_text), len(test_text), len(np.unique(train_y))))
print("class balance:", np.bincount(train_y))
pd.DataFrame({"label": [CLASS_NAMES[i] for i in train_y[:5]], "text": train_text[:5]})

raw: train=120000 test=7600 classes=4
class balance: [30000 30000 30000 30000]


,label,text
0,Business,wall st. bears claw back into the black (reute...
1,Business,carlyle looks toward commercial aerospace (reu...
2,Business,oil and economy cloud stocks' outlook (reuters...
3,Business,iraq halts oil exports from main southern pipe...
4,Business,"oil prices soar to all-time record, posing new..."


In [4]:
def build_vocab(texts, vocab_size):
    counter = collections.Counter()
    for t in texts:
        counter.update(TOKEN_RE.findall(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for token, _ in counter.most_common(vocab_size - 2):
        vocab[token] = len(vocab)
    return vocab

def encode(texts, vocab, max_len):
    unk = vocab["<unk>"]
    out = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(tok, unk) for tok in TOKEN_RE.findall(t)][:max_len]
        out[i, :len(ids)] = ids
    return torch.from_numpy(out)

vocab = build_vocab(train_text, VOCAB_SIZE)
X_all = encode(train_text, vocab, MAX_LEN)
X_test = encode(test_text, vocab, MAX_LEN)
print("vocab=%d  encoded train %s  test %s" % (len(vocab), tuple(X_all.shape), tuple(X_test.shape)))

vocab=30000  encoded train (120000, 64)  test (7600, 64)


In [5]:
def balanced_subset(y, n_total, seed):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    per = n_total // len(classes)
    picks = [rng.choice(np.flatnonzero(y == c), size=min(per, int((y == c).sum())), replace=False)
             for c in classes]
    out = np.concatenate(picks)
    rng.shuffle(out)
    return out

rng = np.random.default_rng(C.SPLIT_SEED)
perm = rng.permutation(len(X_all))
n_val = 12_000
val_idx, train_idx = perm[:n_val], perm[n_val:]

n_sub = C._env_int("FT_TRAIN_SUBSAMPLE", 0)
if n_sub:
    train_idx = train_idx[:n_sub]

X_train, y_train = X_all[train_idx], train_y[train_idx]
X_val, y_val = X_all[val_idx], train_y[val_idx]

# the official test split plays the role of EXP_A's independent fault-analysis
# dataset: a calibration part for single-head criticality and a fixed balanced
# part reused by every random multi-head trial
eval_idx = balanced_subset(test_y, C.N_FAULT, C.FAULT_SPLIT_SEED)
calib_mask = np.ones(len(test_y), dtype=bool)
calib_mask[eval_idx] = False
X_calib, y_calib = X_test[calib_mask], test_y[calib_mask]
X_eval, y_eval = X_test[eval_idx].contiguous(), test_y[eval_idx]

print("train=%d val=%d calib=%d fault_eval=%d"
      % (len(X_train), len(X_val), len(X_calib), len(X_eval)))
print("fault-eval class balance:", np.bincount(y_eval))

train=108000 val=12000 calib=6600 fault_eval=1000
fault-eval class balance: [250 250 250 250]


In [6]:
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.model import TextClassifier, count_params
from faulttol.training import train_classifier

torch.manual_seed(C.TRAIN_SEED)
np.random.seed(C.TRAIN_SEED)

model = TextClassifier(len(vocab), MODEL_CONFIG, max_len=MAX_LEN,
                       num_classes=NUM_CLASSES, activation="relu")
print("params = %d" % count_params(model))
print("heads  = %d layers x %d = %d"
      % (MODEL_CONFIG["num_layers"], MODEL_CONFIG["num_heads"], len(get_all_heads(model))))
model

params = 10840324
heads  = 4 layers x 8 = 32


TextClassifier(
  (pos): PositionalEncoding()
  (layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (W_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.05, inplace=False)
        (3): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
  )
  (norm_out): Identity()
  (dropout): Dropout(p=0.05, inplace=False)
  (embedding): Embedding(30000, 256, padding_idx=0)
  (classifier): Li

In [7]:
from faulttol import wo_faults as W
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.training import load_checkpoint
from scipy.stats import kendalltau, pearsonr, spearmanr

EVAL_CHUNK = 1000
OUT = RESULTS_DIR / "wo_faults"
TABLES, FIGURES = OUT / "tables", OUT / "figures"
for d in (TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
CKPT = RESULTS_DIR / "models" / "best.pt"
SHIPPED = json.loads((RESULTS_DIR / "summary.json").read_text(encoding="utf-8"))

CRIT_COL = "D_pos_mean"   # criticality / damage signal, as in the crash study
DEG_COL = "D_mean"        # degradation variable for the severity analysis
SAFE_COL = "D_pos_mean"   # safe-region degradation variable
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

def save(df, name):
    path = TABLES / name
    df.to_csv(path, index=False)
    print("saved", path.relative_to(ROOT), df.shape)
    return path

print("checkpoint:", CKPT.relative_to(ROOT), "| exists:", CKPT.is_file())
print("artifacts ->", OUT.relative_to(ROOT))
print("N_FAULT=%d  WO_REPEATS=%d  WO_K_TRIALS=%d  WO_SAMPLES=%d"
      % (C.N_FAULT, C.WO_REPEATS, C.WO_K_TRIALS, C.WO_DEEPDIVE_SAMPLES))


checkpoint: results\base_transformer\ag_news\models\best.pt | exists: True
artifacts -> results\base_transformer\ag_news\wo_faults
N_FAULT=1000  WO_REPEATS=10  WO_K_TRIALS=2000  WO_SAMPLES=30


## Load the trained checkpoint

In [8]:
ckpt = load_checkpoint(CKPT, model)
model.to(DEVICE).eval()
CHECKSUM = W.model_checksum(model)
print("loaded epoch %d, best val score %.4f" % (ckpt["epoch"], ckpt["best_score"]))

def evaluate():
    return classification_outputs(model, X_eval, y_eval, DEVICE, chunk=EVAL_CHUNK)

damage_fn = classification_damage
clean_eval = classification_clean_metrics(evaluate())
print("clean (fixed eval subset, n=%d):" % len(X_eval), clean_eval)
if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
    for k, v in clean_eval.items():
        if abs(v - SHIPPED["clean_metrics"][k]) > 1e-9:
            print("WARNING: clean %s %.6f differs from summary.json %.6f - "
                  "is the checkpoint from the same run?" % (k, v, SHIPPED["clean_metrics"][k]))
    print("clean metrics checked against summary.json")

t0 = time.time()
for _ in range(5):
    evaluate()
per_eval = (time.time() - t0) / 5.0
print("%.3f s per evaluation" % per_eval)


loaded epoch 6, best val score 0.9860


clean (fixed eval subset, n=1000): {'accuracy': 0.907, 'precision': 0.9070246341055538, 'recall': 0.9069999999999999, 'f1': 0.906797791029441, 'roc_auc': 0.9845946666666667}
clean metrics checked against summary.json


0.272 s per evaluation


## Crash reference

The single-head crash analysis is re-run on the loaded checkpoint (32
evaluations). It gives the reference criticality, the severity weights `W` and
the critical-head set used by the multi-head study, and picks the deep-dive
heads: the top 3 by crash criticality plus the least critical head.


In [9]:
crash_df = single_head.run_single_head_analysis(
    model=model, evaluate=evaluate, damage_fn=damage_fn,
    criticality_col=CRIT_COL, model_name=MODEL_NAME, verbose=False)
save(crash_df, "crash_single_head.csv")

shipped_sh = pd.read_csv(RESULTS_DIR / "tables" / "single_head_damage.csv")
if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
    diff = np.abs(shipped_sh["criticality"].to_numpy() - crash_df["criticality"].to_numpy()).max()
    print("max |criticality - shipped| = %.2e" % diff,
          "(OK)" if diff < 1e-6 else "<- WARNING: differs from the shipped crash run")

weight_lookup = single_head.build_head_weight_lookup(crash_df)
critical_heads = single_head.critical_head_set(crash_df, C.CRITICAL_TOP_FRACTION)
DEEP_HEADS = W.select_deepdive_heads(crash_df, C.WO_DEEPDIVE_N_CRITICAL)
print("critical heads:", ["L%d_H%d" % h for h in critical_heads])
print("deep-dive heads:", ["L%d_H%d (%s)" % h for h in DEEP_HEADS])
M_LAYER = W.layer_wo_absmax(model)
print("max|W_o| per layer (m):", ["%.4f" % m for m in M_LAYER])
crash_df.nlargest(5, "criticality")[["head_name", "criticality", DEG_COL]]


saved results\base_transformer\ag_news\wo_faults\tables\crash_single_head.csv (32, 30)
max |criticality - shipped| = 9.71e-17 (OK)
critical heads: ['L0_H4', 'L1_H4', 'L1_H2']
deep-dive heads: ['L0_H4 (top1)', 'L1_H4 (top2)', 'L1_H2 (top3)', 'L0_H2 (least)']
max|W_o| per layer (m): ['0.1405', '0.1306', '0.1305', '0.1405']


,head_name,criticality,D_mean
4,L0_H4,0.012335,0.008116
12,L1_H4,0.011205,0.003852
10,L1_H2,0.010637,0.006600
30,L3_H6,0.010269,0.003361
31,L3_H7,0.009770,0.001227


## E0: sanity check

Zeroing a head's `W_o` columns must give **bit-identical** outputs to the crash
mask, for every single head and for 50 random multi-head sets (the first 50
configurations of the crash study). Two more checks exercise the bit-flip code:
a bit error rate of 0, and the same bit flipped twice, must both equal the clean
model.


In [10]:
sanity = W.run_wo_sanity(model, evaluate, n_multi=C.WO_SANITY_MULTI,
                         seed=C.MULTIHEAD_FAULT_SEED, fault_seed=C.WO_FAULT_SEED)
save(sanity, "sanity_checks.csv")
sanity_summary = sanity.groupby("check").agg(n=("exact", "size"), all_exact=("exact", "all"),
                                             max_abs_diff=("max_abs_diff", "max"))
assert sanity["exact"].all(), "W_o fault does not reproduce the expected model state"
assert W.model_checksum(model) == CHECKSUM, "weights were not restored"
print("all sanity checks exact; weights restored")
sanity_summary


saved results\base_transformer\ag_news\wo_faults\tables\sanity_checks.csv (146, 4)
all sanity checks exact; weights restored


,n,all_exact,max_abs_diff
check,,,
ber0_vs_clean,32,True,0.0
double_flip_vs_clean,32,True,0.0
zero_vs_mask_multi,50,True,0.0
zero_vs_mask_single,32,True,0.0


## E1: single-head W_o faults, every head

Each head in turn gets one W_o fault while all other heads stay intact:

- **bit flips** at bit error rate (BER) 1e-6 ... 1e-3: every bit of the head's
  `W_o` columns flips independently with probability BER;
- **stuck-at**: a random fraction of the head's `W_o` weights is stuck at 0,
  at +m or at -m, where m = max|W_o| of the head's layer.

Stochastic levels are repeated `WO_REPEATS` times. The W_o criticality of a head
is the mean `D_pos_mean` over repeats, the same formula as the crash
criticality. `catastrophic_rate` is the share of trials whose outputs became
non-finite (counted as failed predictions, see `docs/wo_faults.md`).


In [11]:
fault_grid = W.build_fault_grid(C.WO_BER_LEVELS, C.WO_STUCK_VALUES_REL, C.WO_STUCK_FRACS)
wo_sh = W.run_wo_single_head(model, evaluate, damage_fn, fault_grid, C.WO_REPEATS,
                             seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
save(wo_sh, "wo_single_head.csv")
assert W.model_checksum(model) == CHECKSUM

wo_crit = W.wo_criticality(wo_sh, CRIT_COL)
save(wo_crit, "wo_single_head_criticality.csv")
crash_cmp = W.compare_with_crash(wo_crit, crash_df)
save(crash_cmp, "wo_vs_crash_single_head.csv")

plots.wo_level_curves(wo_crit, crash_df, "%s - single-head W_o fault criticality" % MODEL_NAME,
                      FIGURES / "wo_single_head_levels.png", ylabel=CRIT_COL)
plots.crash_vs_wo_criticality(wo_crit, crash_df, "%s - crash vs W_o criticality" % MODEL_NAME,
                              FIGURES / "crash_vs_wo_criticality.png")
crash_cmp


    single-head 500/4256  3.4 evals/s  eta 18.3 min


    single-head 1000/4256  3.4 evals/s  eta 15.8 min


    single-head 1500/4256  3.4 evals/s  eta 13.4 min


    single-head 2000/4256  3.4 evals/s  eta 11.0 min


    single-head 2500/4256  3.4 evals/s  eta 8.5 min


    single-head 3000/4256  3.4 evals/s  eta 6.1 min


    single-head 3500/4256  3.4 evals/s  eta 3.7 min


    single-head 4000/4256  3.4 evals/s  eta 1.2 min


saved results\base_transformer\ag_news\wo_faults\tables\wo_single_head.csv (4256, 38)
saved results\base_transformer\ag_news\wo_faults\tables\wo_single_head_criticality.csv (512, 13)


saved results\base_transformer\ag_news\wo_faults\tables\wo_vs_crash_single_head.csv (16, 9)


,fault_model,level,mean_criticality,max_criticality,most_critical_head,ratio_to_crash_mean,spearman_vs_crash,top3_overlap_with_crash,catastrophic_rate
0,bitflip,ber=1e-06,0.004081,0.065280,L1_H3,0.491889,0.186967,0,0.006250
1,bitflip,ber=1e-05,0.030633,0.130560,L2_H4,3.692287,-0.068548,0,0.046875
2,bitflip,ber=0.0001,0.373386,0.587488,L3_H0,45.005864,0.160557,0,0.571875
3,bitflip,ber=0.001,0.652763,0.652763,L0_H0,78.680396,NaN,0,1.000000
4,stuck_at_0,frac=0.01,0.000183,0.000290,L0_H0,0.022075,0.175220,1,0.000000
5,stuck_at_0,frac=0.1,0.000888,0.001242,L0_H4,0.106995,0.568915,1,0.000000
6,stuck_at_0,frac=0.5,0.003874,0.005223,L0_H4,0.466921,0.923021,3,0.000000
7,stuck_at_0,frac=1,0.008296,0.012335,L0_H4,1.000000,1.000000,3,0.000000
8,stuck_at_+m,frac=0.01,0.000603,0.001072,L0_H6,0.072728,0.163123,1,0.000000
9,stuck_at_+m,frac=0.1,0.002023,0.003381,L0_H4,0.243896,0.243768,1,0.000000


## E2: random multi-head W_o faults

The head sets are drawn with the crash study's seed, so trial *k* here faults
exactly the heads that crash trial *k* crashed; only the fault differs. Each
failed head gets the W_o fault of the setting. The analysis is the crash
study's: degradation grid, rank correlations of severity vs degradation (plus
the size of the weight change `rel_change` and the number of flipped bits), and
the 95% safe regions.


In [12]:
def severity_correlations(df, deg_col, severities):
    rows = []
    for sev in severities:
        x, y = df[sev].to_numpy(float), df[deg_col].to_numpy(float)
        fin = np.isfinite(x) & np.isfinite(y)
        if np.std(x[fin]) == 0:
            continue
        pr = pearsonr(x[fin], y[fin]).statistic
        sr, sp = spearmanr(x, y)        # ranks: inf is just the largest value
        kr = kendalltau(x, y).statistic
        rows.append({"severity": sev, "degradation": deg_col, "pearson_finite": pr,
                     "spearman": sr, "kendall": kr, "spearman_p": sp})
    return pd.DataFrame(rows)

n_heads = len(get_all_heads(model))
crash_mh = pd.read_csv(RESULTS_DIR / "tables" / "random_multihead.csv")
mh_frames, mh_summary, curves = {}, [], {}
if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
    k = min(C.WO_K_TRIALS, len(crash_mh))
    curves["crash (shipped, same head sets)"] = (crash_mh["N_fail"][:k], crash_mh[DEG_COL][:k])

for setting, spec in C.WO_MULTIHEAD_SETTINGS:
    print("setting", setting)
    df = W.run_wo_multihead(model, evaluate, damage_fn, spec, setting, weight_lookup,
                            critical_heads, MODEL_NAME, C.WO_K_TRIALS,
                            seed=C.MULTIHEAD_FAULT_SEED, fault_seed=C.WO_FAULT_SEED,
                            num_layers=model.num_layers)
    assert W.model_checksum(model) == CHECKSUM
    mh_frames[setting] = df
    save(df, "wo_multihead_%s.csv" % setting)

    grid = multihead.degradation_grid(df, DEG_COL)
    save(grid.reset_index(), "wo_degradation_grid_%s.csv" % setting)
    plots.degradation_grid_plot(grid, "%s - W_o %s: mean %s" % (MODEL_NAME, setting, DEG_COL),
                                FIGURES / ("wo_degradation_grid_%s.png" % setting),
                                value_label=DEG_COL)
    corr = severity_correlations(df, DEG_COL, ("N_fail", "L_fault", "N_crit_fail", "W_fail",
                                               "W_fail_mean", "rel_change", "n_bitflips",
                                               "n_exp_msb_flips"))
    save(corr, "wo_severity_correlations_%s.csv" % setting)
    grids, safe = saferegion.run_safe_region_analysis(
        df, SAFE_COL, TAUS, n_heads, len(critical_heads),
        min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB, tau_labels=TAU_LABELS)
    save(grids, "wo_safe_region_grid_%s.csv" % setting)
    save(safe, "wo_safe_region_summary_%s.csv" % setting)
    curves["W_o " + setting] = (df["N_fail"], df[DEG_COL])

    row = {"setting": setting, "k_trials": len(df),
           "mean_" + DEG_COL: float(df[DEG_COL].mean()),
           "catastrophic_trial_rate": float((df["nonfinite_rate"] > 0).mean()),
           "mean_nonfinite_sample_rate": float(df["nonfinite_rate"].mean()),
           "spearman_N_fail": float(corr.set_index("severity").loc["N_fail", "spearman"]),
           "safe_regions": safe.to_dict(orient="records")}
    if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
        paired = crash_mh[DEG_COL].to_numpy()[:len(df)]
        row["crash_mean_" + DEG_COL + "_same_sets"] = float(paired.mean())
        row["share_trials_worse_than_crash"] = float(np.mean(df[DEG_COL].to_numpy() > paired))
    mh_summary.append(row)

plots.degradation_curves(curves, "%s - degradation vs failed heads: crash vs W_o faults" % MODEL_NAME,
                         FIGURES / "wo_multihead_curves.png", ylabel="mean " + DEG_COL)
pd.DataFrame(mh_summary).drop(columns="safe_regions")


setting bitflip_ber1e-4


    bitflip_ber1e-4 500/2000  3.0 evals/s  eta 8.4 min


    bitflip_ber1e-4 1000/2000  3.0 evals/s  eta 5.5 min


    bitflip_ber1e-4 1500/2000  3.0 evals/s  eta 2.8 min


    bitflip_ber1e-4 2000/2000  3.0 evals/s  eta 0.0 min


saved results\base_transformer\ag_news\wo_faults\tables\wo_multihead_bitflip_ber1e-4.csv (2000, 45)
saved results\base_transformer\ag_news\wo_faults\tables\wo_degradation_grid_bitflip_ber1e-4.csv (5, 5)
saved results\base_transformer\ag_news\wo_faults\tables\wo_severity_correlations_bitflip_ber1e-4.csv (8, 6)


... (4 outputs left out)

    stuck_+m_frac0.10 2000/2000  2.9 evals/s  eta 0.0 min


saved results\base_transformer\ag_news\wo_faults\tables\wo_multihead_stuck_+m_frac0.10.csv (2000, 45)
saved results\base_transformer\ag_news\wo_faults\tables\wo_degradation_grid_stuck_+m_frac0.10.csv (5, 5)


saved results\base_transformer\ag_news\wo_faults\tables\wo_severity_correlations_stuck_+m_frac0.10.csv (6, 6)
saved results\base_transformer\ag_news\wo_faults\tables\wo_safe_region_grid_stuck_+m_frac0.10.csv (372, 9)
saved results\base_transformer\ag_news\wo_faults\tables\wo_safe_region_summary_stuck_+m_frac0.10.csv (3, 9)


,setting,k_trials,mean_D_mean,catastrophic_trial_rate,mean_nonfinite_sample_rate,spearman_N_fail,crash_mean_D_mean_same_sets,share_trials_worse_than_crash
0,bitflip_ber1e-4,2000,0.622893,0.9745,0.9745,0.257316,0.149042,0.9745
1,stuck_+m_frac0.10,2000,0.004513,0.0000,0.0000,0.840462,0.149042,0.0055


## Deep-dive on the most critical heads

The heads are the top 3 by crash criticality plus the least critical head as a
baseline. Every trial changes the `W_o` columns of **one** head and nothing
else. `m` is max|W_o| of the head's layer.


### D1: damage vs the value a weight is changed to

Each value in the grid is written to:

- **single**: 50 random single weights, one at a time;
- **row**: 8 random whole rows (the `d_k` weights that write one output feature);
- **column**: 8 random whole columns (the `d_model` weights that read one of the
  head's `d_k` output dimensions);
- **block**: all of the head's `W_o` weights.

The same targets are reused for every value, so values are compared on paired
targets. Expect the column and block curves to be **flat**: a constant column
adds the same number to every output feature, and the post-norm LayerNorm
subtracts it again, so any constant there acts exactly like zero.


In [13]:
d1 = W.run_value_sweep(model, evaluate, damage_fn, DEEP_HEADS, C.WO_VALUE_GRID,
                       n_single=C.WO_SWEEP_SINGLE, n_columns=C.WO_SWEEP_COLUMNS,
                       n_rows=C.WO_SWEEP_ROWS, seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
assert W.model_checksum(model) == CHECKSUM
save(d1, "deep_value_sweep.csv")
d1_table = d1.pivot_table(index=["granularity", "value_label"], columns="head_name",
                          values=CRIT_COL, aggfunc="mean", sort=False)
save(d1_table.reset_index(), "deep_value_sweep_summary.csv")
plots.value_sweep_plot(d1, CRIT_COL, "%s - D1: damage vs the value W_o weights are set to"
                       % MODEL_NAME, FIGURES / "deep_value_sweep.png")
d1_table


    value sweep 500/3484  3.5 evals/s  eta 14.1 min


    value sweep 1000/3484  3.5 evals/s  eta 11.7 min


    value sweep 1500/3484  3.5 evals/s  eta 9.4 min


    value sweep 2000/3484  3.5 evals/s  eta 7.0 min


    value sweep 2500/3484  3.5 evals/s  eta 4.6 min


    value sweep 3000/3484  3.5 evals/s  eta 2.3 min


saved results\base_transformer\ag_news\wo_faults\tables\deep_value_sweep.csv (3484, 42)
saved results\base_transformer\ag_news\wo_faults\tables\deep_value_sweep_summary.csv (52, 6)


head_name                   L0_H4     L1_H4     L1_H2     L0_H2
granularity value_label                                        
single      0            0.000019  0.000014  0.000011  0.000018
            +m/2         0.000042  0.000028  0.000027  0.000032
            -m/2         0.000043  0.000028  0.000022  0.000045
            +m           0.000084  0.000056  0.000052  0.000071
            -m           0.000084  0.000055  0.000046  0.000085
            +10m         0.000906  0.000570  0.000504  0.000784
            -10m         0.000848  0.000573  0.000491  0.000809
            +1e2         0.298045  0.240215  0.209411  0.294167
            -1e2         0.301521  0.241959  0.205686  0.290551
            +1e4         0.660651  0.657033  0.653092  0.660464
            -1e4         0.660476  0.651390  0.655083  0.659751
            +1e6         0.661606  0.662248  0.658057  0.661526
            -1e6         0.661305  0.655478  0.660045  0.660813
row         0            0.000167  0.000141  0.000112  0.000150
            +m/2         0.000285  0.000237  0.000175  0.000274
            -m/2         0.000257  0.000220  0.000214  0.000246
            +m           0.000493  0.000389  0.000302  0.000469
            -m           0.000437  0.000370  0.000365  0.000437
            +10m         0.004773  0.003815  0.003274  0.004924
            -10m         0.004736  0.003831  0.003745  0.004895
            +1e2         0.607081  0.586151  0.577781  0.615278
            -1e2         0.607303  0.579270  0.579747  0.616736
            +1e4         0.662295  0.658321  0.659432  0.657372
            -1e4         0.658302  0.652614  0.661211  0.662079
            +1e6         0.662777  0.659238  0.660123  0.657435
            -1e6         0.658826  0.653515  0.662065  0.662355
column      0            0.000630  0.000617  0.000519  0.000618
            +m/2         0.000630  0.000617  0.000519  0.000618
            -m/2         0.000630  0.000617  0.000519  0.000618
            +m           0.000630  0.000617  0.000519  0.000618
            -m           0.000630  0.000617  0.000519  0.000618
            +10m         0.000630  0.000617  0.000519  0.000618
            -10m         0.000630  0.000617  0.000519  0.000618
            +1e2         0.000631  0.000617  0.000519  0.000618
            -1e2         0.000631  0.000617  0.000519  0.000618
            +1e4         0.000633  0.000618  0.000520  0.000619
            -1e4         0.000633  0.000618  0.000520  0.000619
            +1e6         0.003269  0.002909  0.003190  0.003549
            -1e6         0.003233  0.002904  0.003188  0.003576
block       0            0.012335  0.011205  0.010637  0.005102
            +m/2         0.012335  0.011205  0.010637  0.005102
            -m/2         0.012335  0.011205  0.010637  0.005102
            +m           0.012335  0.011205  0.010637  0.005102
            -m           0.012335  0.011205  0.010637  0.005102
            +10m         0.012335  0.011205  0.010637  0.005102
            -10m         0.012335  0.011205  0.010637  0.005102
            +1e2         0.012334  0.011205  0.010637  0.005102
            -1e2         0.012334  0.011205  0.010637  0.005102
            +1e4         0.012299  0.011188  0.010588  0.005142
            -1e4         0.012305  0.011193  0.010589  0.005134
            +1e6         0.019703  0.018327  0.022007  0.015610
            -1e6         0.019983  0.017789  0.021857  0.015789

### D2: damage vs the original magnitude of the changed weight

The head's weights are ranked by |w| into deciles (`d01` smallest ... `d10`
largest) plus the top 1% and top 0.1%. From each bin, single weights are
**zeroed**, **negated**, **multiplied by 10** or get their **exponent MSB
flipped**, one weight per trial. Finally each whole bin is zeroed at once: is
removing the 10% largest weights worse than removing the 10% smallest?


In [14]:
d2 = W.run_magnitude_study(model, evaluate, damage_fn, DEEP_HEADS, C.WO_DEEPDIVE_SAMPLES,
                           seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
assert W.model_checksum(model) == CHECKSUM
save(d2, "deep_magnitude.csv")
d2_table = d2.pivot_table(index=["granularity", "op", "magnitude_bin"], columns="head_name",
                          values=CRIT_COL, aggfunc="mean", sort=False)
save(d2_table.reset_index(), "deep_magnitude_summary.csv")
plots.magnitude_bins_plot(d2, CRIT_COL, "%s - D2: damage vs |w| of the changed weight"
                          % MODEL_NAME, FIGURES / "deep_magnitude.png")

# rank correlation between |w| and damage, per head and operation (single weights).
# NaN means the damage was constant, e.g. every exponent-MSB flip is catastrophic.
import warnings
d2_single = d2[d2["granularity"] == "single"].copy()
d2_single["abs_old"] = d2_single["old_value"].abs()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d2_corr = (d2_single.groupby(["head_name", "role", "op"])
               .apply(lambda g: spearmanr(g["abs_old"], g[CRIT_COL]).statistic,
                      include_groups=False)
               .rename("spearman_abs_w_vs_damage").reset_index())
save(d2_corr, "deep_magnitude_correlation.csv")
d2_corr.pivot(index=["head_name", "role"], columns="op", values="spearman_abs_w_vs_damage")


    magnitude 500/5472  3.6 evals/s  eta 23.3 min


    magnitude 1000/5472  3.6 evals/s  eta 21.0 min


    magnitude 1500/5472  3.6 evals/s  eta 18.6 min


    magnitude 2000/5472  3.6 evals/s  eta 16.3 min


    magnitude 2500/5472  3.6 evals/s  eta 13.9 min


    magnitude 3000/5472  3.6 evals/s  eta 11.6 min


... (3 outputs left out)

    magnitude 5000/5472  3.6 evals/s  eta 2.2 min


saved results\base_transformer\ag_news\wo_faults\tables\deep_magnitude.csv (5472, 44)
saved results\base_transformer\ag_news\wo_faults\tables\deep_magnitude_summary.csv (60, 7)


saved results\base_transformer\ag_news\wo_faults\tables\deep_magnitude_correlation.csv (16, 4)


,op,flip_exp_msb,negate,x10,zero
head_name,role,,,,
L0_H2,least,NaN,0.971495,0.967476,0.971542
L0_H4,top1,NaN,0.948506,0.956155,0.948672
L1_H2,top3,NaN,0.957379,0.949085,0.957334
L1_H4,top2,NaN,0.953249,0.959622,0.953072


### D3: damage vs the float32 bit that is flipped

For every bit position 0..31 (0-22 mantissa, 23-30 exponent with 30 the MSB, 31
sign) the same random weights get that one bit flipped, one weight per trial.


In [15]:
d3 = W.run_bit_position_study(model, evaluate, damage_fn, DEEP_HEADS, C.WO_DEEPDIVE_SAMPLES,
                              seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
assert W.model_checksum(model) == CHECKSUM
save(d3, "deep_bit_position.csv")
d3_table = d3.groupby(["head_name", "role", "bit"]).agg(
    damage_mean=(CRIT_COL, "mean"), damage_max=(CRIT_COL, "max"),
    catastrophic_rate=("nonfinite_rate", lambda s: float(np.mean(s > 0)))).reset_index()
save(d3_table, "deep_bit_position_summary.csv")
plots.bit_position_plot(d3, CRIT_COL, "%s - D3: damage per flipped bit of a W_o weight"
                        % MODEL_NAME, FIGURES / "deep_bit_position.png")
d3.groupby(["bit_field", "role"]).agg(
    damage_mean=(CRIT_COL, "mean"),
    catastrophic_rate=("nonfinite_rate", lambda s: float(np.mean(s > 0)))).unstack()


    bit position 500/3840  3.5 evals/s  eta 15.7 min


    bit position 1000/3840  3.5 evals/s  eta 13.4 min


    bit position 1500/3840  3.5 evals/s  eta 11.0 min


    bit position 2000/3840  3.5 evals/s  eta 8.7 min


    bit position 2500/3840  3.5 evals/s  eta 6.3 min


    bit position 3000/3840  3.5 evals/s  eta 4.0 min


    bit position 3500/3840  3.5 evals/s  eta 1.6 min


saved results\base_transformer\ag_news\wo_faults\tables\deep_bit_position.csv (3840, 41)
saved results\base_transformer\ag_news\wo_faults\tables\deep_bit_position_summary.csv (128, 6)


damage_mean                                            \
role              least          top1          top2          top3   
bit_field                                                           
exponent   8.165568e-02  8.165978e-02  8.163524e-02  8.163906e-02   
mantissa   5.157663e-07  6.270651e-07  3.967456e-07  5.089615e-07   
sign       3.246301e-05  4.331061e-05  2.548480e-05  3.321648e-05   

          catastrophic_rate                       
role                  least   top1   top2   top3  
bit_field                                         
exponent              0.125  0.125  0.125  0.125  
mantissa              0.000  0.000  0.000  0.000  
sign                  0.000  0.000  0.000  0.000

### D4: is damage explained by the size of the change?

All deep-dive trials pooled, as damage against `||dW||_F`, the Frobenius norm
of the change to the head's `W_o` columns. If the curves of the critical heads
lie above the least critical head at the same `||dW||`, *where* the change
lands matters, not only how large it is.


In [16]:
pooled = W.pooled_delta_table({"value_sweep": d1, "magnitude": d2, "bit_position": d3}, CRIT_COL)
binned = W.binned_damage_by_delta(pooled, CRIT_COL)
save(binned, "deep_damage_vs_delta.csv")
plots.delta_damage_plot(pooled, binned, CRIT_COL,
                        "%s - D4: damage vs size of the W_o change" % MODEL_NAME,
                        FIGURES / "deep_damage_vs_delta.png")
fin = pooled[np.isfinite(pooled["delta_norm"])]
d4_corr = fin.groupby("role").apply(
    lambda g: spearmanr(g["delta_norm"], g[CRIT_COL]).statistic, include_groups=False)
print("Spearman(||dW||, damage) by head role (finite changes):")
print(d4_corr.round(3).to_string())


saved results\base_transformer\ag_news\wo_faults\tables\deep_damage_vs_delta.csv (34, 7)


Spearman(||dW||, damage) by head role (finite changes):
role
least    0.986
top1     0.984
top2     0.989
top3     0.989


## Summary

In [17]:
def _top_findings():
    by_field = d3.groupby("bit_field").agg(damage_mean=(CRIT_COL, "mean"),
        catastrophic_rate=("nonfinite_rate", lambda s: float(np.mean(s > 0))))
    return {
        "bit_position_by_field": by_field.to_dict(orient="index"),
        "damage_vs_delta_spearman_by_role": {k: float(v) for k, v in d4_corr.items()},
        "magnitude_spearman": d2_corr.to_dict(orient="records"),
    }

summary = {
    "model_name": MODEL_NAME,
    "checkpoint": str(CKPT.relative_to(ROOT)),
    "fault_location": "W_o.weight[:, h*d_k:(h+1)*d_k] (the head's columns of the output projection)",
    "n_fault_eval_samples": int(C.N_FAULT),
    "clean_metrics": clean_eval,
    "seconds_per_evaluation": per_eval,
    "parameters": {
        "wo_repeats": C.WO_REPEATS, "wo_k_trials": C.WO_K_TRIALS,
        "ber_levels": list(C.WO_BER_LEVELS), "stuck_values_rel": list(C.WO_STUCK_VALUES_REL),
        "stuck_fracs": list(C.WO_STUCK_FRACS),
        "multihead_settings": [{"setting": s, "spec": sp} for s, sp in C.WO_MULTIHEAD_SETTINGS],
        "value_grid": [list(v) for v in C.WO_VALUE_GRID],
        "sweep_single": C.WO_SWEEP_SINGLE, "sweep_rows": C.WO_SWEEP_ROWS,
        "sweep_columns": C.WO_SWEEP_COLUMNS,
        "deepdive_samples": C.WO_DEEPDIVE_SAMPLES, "fault_seed": C.WO_FAULT_SEED,
        "m_layer": M_LAYER,
    },
    "deep_dive_heads": [{"head": "L%d_H%d" % (l, h), "role": r,
                         "crash_criticality": float(crash_df.loc[(crash_df["layer"] == l)
                                                                 & (crash_df["head"] == h),
                                                                 "criticality"].iloc[0])}
                        for l, h, r in DEEP_HEADS],
    "sanity": sanity_summary.reset_index().to_dict(orient="records"),
    "single_head_vs_crash": crash_cmp.to_dict(orient="records"),
    "multihead": mh_summary,
    **_top_findings(),
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
assert W.model_checksum(model) == CHECKSUM
print("wrote", (OUT / "summary.json").relative_to(ROOT))


wrote results\base_transformer\ag_news\wo_faults\summary.json
